# Лабораторная работа 2

Цель работы: освоить основные подходы к решению задачи анализа тональности (sentiment analysis) для текстов на русском языке.

## Анализ тональности (Sentiment Analysis)

Анализ тональности - это задача определения эмоциональной окраски текста (например, положительная, отрицательная, нейтральная).

Модель должна понять, какое отношение выражает автор текста — например, в отзывах, комментариях, новостях или постах в соцсетях.
Примеры:

«Фильм просто великолепен!» - положительная тональность

«Обслуживание ужасное!» - отрицательная тональность

«Доставка заняла два дня.» - нейтральная тональность


In [ ]:
# Импорт необходимых библиотек
# Успешное выполнение этой ячейки кода подтверждает правильную настройку среды разработки

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import re
import nltk
from nltk.corpus import stopwords
import string
from tqdm.auto import tqdm
import os

nltk.download('stopwords')

**Загрузка данных:**

Краткое описание датасета:

"**Обзор**:

Этот набор данных представляет собой обширную коллекцию русскоязычных обзоров и текстов, аннотированных для анализа тональности. Он предназначен для поддержки разработки и оценки моделей анализа настроений для русского языка. Набор данных сбалансирован по классам настроений (neutral, positive, negative), чтобы обеспечить справедливое обучение модели и ее оценку.

**Содержание**:

Набор данных составлен на основе нескольких общедоступных источников. Все исходные наборы данных были тщательно сбалансированы по классам тональности (нейтральная, положительная и отрицательная), чтобы обеспечить надёжность и непредвзятость набора данных.

**Структура набора данных**

Набор данных содержит следующие столбцы:

text: Текст рецензии или комментария.
label: Метка тональности, где:
0: нейтральная
1: положительная
2: отрицательная
src: Исходный набор данных."

Источник: https://www.kaggle.com/datasets/mar1mba/russian-sentiment-dataset

Загружаем данные из файла sentiment_dataset.csv в dataframe. Оставим только данные из набора с отзывами.

In [ ]:
# Загрузка данных из sentiment_dataset.csv
print("Загрузка данных из файла 'sentiment_dataset.csv'...")

try:
    df = pd.read_csv('sentiment_dataset.csv')

    # Преобразование числовых меток в строковые для удобства
    sentiment_map = {0: 'neutral', 1: 'positive', 2: 'negative'}
    df['sentiment'] = df['label'].map(sentiment_map)

    # Оставим только отзывы из набора данных
    df = df[df['src'] == 'rureviews']

    print("Данные успешно загружены из 'sentiment_dataset.csv'. Первые 5 строк:")
    display(df.head())
    print("\nИнформация о данных:")
    df.info()
    print("\nРаспределение классов:")
    display(df['sentiment'].value_counts())

except FileNotFoundError:
    print("Файл данных 'sentiment_dataset.csv' не найден в текущей директории.")
except Exception as e:
    print(f"Произошла ошибка при загрузке или обработке данных: {e}")

In [ ]:
# Оставим только нужные нам значения
df = df[['text', 'sentiment', 'label']]

print(f'Таблца до проверки содержит {df.shape[0]} строк.\n')

# Удаляем строки с пропущенными значениями (если есть)
df = df.dropna()

print(f'Таблца без пропущенных значений содержит {df.shape[0]} строк.\n')

# Удаляем повторяющиеся отзывы (если есть)
duplicated_mask = df.duplicated(subset = 'text', keep = False)

print(f'Всего {np.sum(duplicated_mask)} повторяющихся отзывов в наборе данных.\n')

df = df.drop_duplicates(subset = 'text', keep = 'first')

print(f'Таблца без повторяющихся отзывов содержит {df.shape[0]} строк.\n')

Мы видим пример предварительно обработанного датасета: отсутствуют повторы и пропущенные значения в таблице. На практике всегда лучше проверить этот момент.

# Задание 1

Оставьте по 5000 строк для каждого класса в датасете (neutral, positive,negative). Всего должно получиться 15000 примеров. (Используйте sample())

Мы сильно сокращаем датасет для лабораторной работы, чтобы время обработки данных и обучения моделей было приемлемым. Однако на практике чем больше у нас разнообразных данных для обучения моделей, тем лучше результаты мы получм, если правильно подберём методы и настроим гиперпараметры.

In [ ]:
# Сэмплирование данных: оставляем по 5000 строк для каждого уникального значения sentiment
print("Сэмплирование данных: оставляем по 5000 строк для каждого класса")

#### ВСТАВЬТЕ КОД СЮДА
df = ...
####

# Сбрасываем индекс
df.reset_index(drop=True, inplace=True)


print("\nИнформация о сэмплированных данных:\n")
df.info()
print("\nРаспределение классов после сэмплирования:\n")
display(df['sentiment'].value_counts())

In [ ]:
df

Уже сейчас можно заметить, что для некоторых примеров однозначно определить настроение автора довольно сложно. Кроме того, для отзывов метка класса часто присваивается автоматически (датасет не размечается вручную), исходя из оценки, которую пользователь поставил товару или услуге. И эта оценка не всегда совпадает с содержанием отзыва (например, человек поставил 5 звёзд из 5 за товар, но в комментарии указал только то, что ему при этом не понравилось).

# Задание 2

**Предобработка текста:**

Проведите предварительную обработку текста в функции preprocess_text()

*   Приведение к нижнему регистру: Преобразуйте все слова в нижний регистр.
*   Удаление знаков пунктуации и специальных символов.
*   Удаление стоп-слов: Используйте список стоп-слов для русского языка для удаления часто встречающихся, но не несущих важную информацию для анализа слов.
*   Лемматизация: приведите слова к нормальной форме

In [ ]:

#### ВСТАВЬТЕ КОД СЮДА

def preprocess_text(text):
    ...
    return " ".join(lemmas)

####

print("\nНачало предобработки текста...")
# Применение предобработки к столбцу с текстом. Используем tqdm для отслеживания прогресса.
if 'text' in df.columns:
    tqdm.pandas()
    df['processed_text'] = df['text'].progress_apply(preprocess_text)
    print("Предобработка текста завершена.")
    display(df.head())

# Задание 3

**Извлечение признаков:**
(Токенизация и векторизация текстов)

Преобразуйте предобработанный текст в числовое представление, пригодное для обучения модели. Используйте метод TF-IDF (Term Frequency-Inverse Document Frequency), реализованнный в классе TfidfVectorizer. В соответствии с вашим вариантом установите количество самых часто встречающихся слов из формируемого словаря VOCABULARY_SIZE, которые будут использоваться при векторизации ("количество слов в словаре").


**Разделение данных:**

Разделите данные на тренировочную и тестовую выборки в соответствии с вашим вариантом ("процент тестовой выборки от объёма всего датасета").

In [ ]:
#### ВСТАВЬТЕ КОД СЮДА

VOCABULARY_SIZE = ...
# Подготовка данных для модели


X = ...
y = ...


# Разделение данных на обучающую и тестовую выборки
# Используйте stratify=y для сохранения распределения классов
X_train, X_test, y_train, y_test = ...
print(f"\nДанные разделены на обучающую ({len(X_train)} примеров) и тестовую ({len(X_test)} примеров) выборки.")
print("Распределение классов в обучающей выборке:")
display(y_train.value_counts())
print("Распределение классов в тестовой выборке:")
display(y_test.value_counts())


# Извлечение признаков с помощью TF-IDF
tfidf_vectorizer = ...
X_train_tfidf = ...
X_test_tfidf = ...

####
print(f"TF-IDF признаки извлечены. Размерность обучающей выборки: {X_train_tfidf.shape}")

# Задание 4

**Обучение модели:**

Обучите модель классификации на тренировочной выборке.

В зависимости от вашего варианта используйте наивный байесовский классификатор (`sklearn.naive_bayes.MultinomialNB`), логистическую регрессию (`sklearn.linear_model.LogisticRegression`), или метод опорных векторов с линейным ядром (`sklearn.svm.LinearSVC`).


In [ ]:

#### ВСТАВЬТЕ КОД СЮДА

model = ...

# Оценка модели (предсказание)
y_pred = ...

####

**Оценка модели:**
Оцените производительность обученной модели на тестовой выборке. Рассчитайте метрики: точность (accuracy), полнота (recall) и F1-мера для каждого класса (положительный, отрицательный, нейтральный), а также общую точность.

**Интерпретация результатов:** Проанализируйте полученные метрики. Попробуйте найти примеры отзывов, которые были классифицированы правильно и неправильно. Объясните, почему модель могла ошибиться в конкретных случаях.

In [ ]:
# Используем zero_division=0 для корректного расчета метрик при отсутствии предсказаний для класса
# Получаем все уникальные метки из набора данных для корректного расчета метрик
all_labels = ['neutral', 'positive', 'negative']
precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average=None, labels=all_labels, zero_division=0) # Расчет по классам
accuracy = accuracy_score(y_test, y_pred)

print("\nРезультаты оценки модели:")
print(f"Общая точность (Accuracy): {accuracy:.4f}")

# Вывод метрик по классам
metrics_df = pd.DataFrame({
    'Sentiment': all_labels,
    'Precision': precision,
    'Recall': recall,
    'F1-score': f1
})
print("\nМетрики по классам:")
display(metrics_df)

# Примеры предсказаний
print("\nПримеры предсказаний:")
# Выбираем примеры случайным образом из тестовой выборки для демонстрации
example_indices = X_test.sample(5).index
examples = df.loc[example_indices]
examples['predicted_sentiment'] = model.predict(tfidf_vectorizer.transform(examples['processed_text']))

for index, row in examples.iterrows():
    print(f"\nОригинальный отзыв: {row['text']}")
    print(f"Предобработанный отзыв: {row['processed_text']}")
    print(f"Истинная тональность: {row['sentiment']}")
    print(f"Предсказанная тональность: {row['predicted_sentiment']}")

# LSTM модель в анализе тональности

LSTM (Long Short-Term Memory) используют в анализе тональности, потому что она умеет:
- понимать контекст и порядок слов,
- запоминать важные слова в длинных предложениях,
- игнорировать несущественные,
- учитывать зависимость между словами, находящимися далеко друг от друга.

**Тональность — это контекстная задача**

Чтобы определить эмоциональную окраску текста, модель должна понимать смысл фразы целиком, а не просто набор слов.

Например:

«Фильм не плохой» - положительная тональность

«Фильм не только скучный, но и затянутый» - отрицательная

Здесь ключевую роль играет слово “не” и его положение относительно прилагательного.

Обычные модели вроде TF-IDF не понимают контекст, в отличие от LSTM.

**LSTM обрабатывает текст как последовательность**

LSTM — это тип рекуррентной нейросети (RNN), которая читает текст по одному слову за раз и запоминает, что было раньше.

Пример:

«Фильм был скучным, но актёры великолепны.»

Чтобы правильно определить тональность, модель должна помнить, что вторая часть фразы меняет общий смысл.

LSTM хранит эту информацию во внутренней памяти — state.

**Проблема «долгой памяти» и её решение**

Обычные RNN не умеют сохранять контекст на длинных промежутках —
градиенты затухают (vanishing gradient).

LSTM решает это с помощью механизма ячеек памяти и трёх "вентилей":

Вентиль	 -  Что делает

* Forget gate	 -  решает, какую старую информацию забыть
* Input gate	 -  решает, что добавить в память
* Output gate	 -  решает, какую часть памяти передать дальше

Благодаря этому LSTM может запомнить даже дальние зависимости:
в предложении «Хотя фильм местами скучный, в целом он отличный» модель понимает, что тональность положительная, несмотря на слово «скучный».

**Работа с последовательностями переменной длины**

В текстах отзывы бывают короткими и длинными.
LSTM умеет работать с последовательностями разной длины (через pad_sequences),
что делает её гибкой для реальных данных.

## Векторизация: подход для сохранения порядка слов
Ранее мы пользовались методом `texts_to_matrix()` класса `Tokenizer()` и его аналогом у `TfidfVectorizer` `transform()`. С помощью этого метода мы получаем матрицу признаков размера [num_texts, num_words] - document-term matrix. То есть каждый документ кодируется вектором фиксированной длины, равной длине используемого для кодирования словаря.

Теперь нас будет интересовать метод `texts_to_sequences()`, который преобразует каждый текст в последовательность индексов слов (список номеров последовательных токенов). То есть длина векторов будет напрямую зависеть от длины документов (отзывов). Этот метод сохраняет **порядок слов**.
Каждому слову присваивается индекс из словаря Tokenizer.word_index.

`texts_to_sequences()` используется для:

*   LSTM / GRU
*   Embedding слоя
*   Моделей, где важен порядок слов

И не подходит для полносвязных сетей без Embedding, так как размерность последовательностей может отличаться (нужно дополнительно применять pad_sequences).


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Сделаем задачу бинарной классификации для упрощения, уберём нейтральные отзывы
texts = df[df['sentiment'] != 'neutral'].processed_text

tokenizer = Tokenizer(num_words=VOCABULARY_SIZE, filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n',
                      lower=True, split=' ', oov_token='unknown', char_level=False)
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)


In [ ]:
texts[0]

In [ ]:
sequences[0]

In [ ]:
# Находим максимальную длину последовательности
max_len = max(len(seq) for seq in sequences)
print(f"Максимальная длина последовательности: {max_len}")

In [ ]:
# Заполняем нулями вектор в конце отзыва, чтобы сделать входные векторы одинаковой длины
X = pad_sequences(sequences, maxlen=max_len, padding='post')

print("Пример последовательности:", X[0])
print("Размер X:", X.shape)

In [ ]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras import utils

y_df = df[df['sentiment'] != 'neutral'].sentiment
y = list(y_df.values)

y = y_df.factorize()[0]

In [ ]:
y

In [ ]:
# Делим на тренировочную и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Архитектура глубокой сети в TensorFlow Keras (продолжение)

### Embedding слой в LSTM

**Что такое Embedding слой?**

Embedding слой в моделях для обработки естественного языка используется для преобразования дискретных входных данных (слов) в плотные векторы фиксированного размера. Вместо того чтобы представлять каждое слово как отдельный уникальный элемент (например, в one-hot кодировании), Embedding слой отображает слова в многомерное векторное пространство, где слова с похожим значением или контекстом находятся ближе друг к другу.

**Зачем он нужен в LSTM?**

В контексте LSTM (и других рекуррентных нейронных сетей) Embedding слой выполняет несколько важных функций:

1.  **Снижение размерности:** One-hot кодирование для большого словаря приводит к очень разреженным и высокоразмерным векторам. Embedding слой позволяет представить слова в гораздо более низкоразмерном пространстве, что делает обучение более эффективным и снижает вычислительные затраты.
2.  **Захват семантических связей:** В процессе обучения Embedding слой изучает векторные представления слов, которые отражают их семантические и синтаксические отношения.
3.  **Улучшение производительности:** Плотные векторные представления, генерируемые Embedding слоем, лучше подходят для входных данных LSTM, поскольку они предоставляют более богатую и информативную репрезентацию слов по сравнению с разреженными представлениями.

**Как задается в TensorFlow (Keras)?**

Embedding слой в TensorFlow Keras задается с помощью класса `tf.keras.layers.Embedding`. Основные параметры:

*   `input_dim`: Размер словаря (общее количество уникальных слов + 1 для зарезервированных индексов, например, для паддинга или неизвестных слов). Уже учитываем это при векторизации.
*   `output_dim`: Размерность выходного вектора (размерность эмбеддинга). Это гиперпараметр, который нужно настраивать.
*   `input_length`: Максимальная длина входных последовательностей (текстов).

### LSTM слой

**Что такое LSTM?**

LSTM (Long Short-Term Memory) - это особый тип рекуррентной нейронной сети (RNN), предназначенный для обработки последовательных данных, таких как текст, речь или временные ряды. Ключевое отличие LSTM от простых RNN заключается в наличии специальной структуры - **ячейки памяти (cell state)** и **вентилей (gates)**, которые позволяют LSTM эффективно учиться и запоминать зависимости на длинных последовательностях, преодолевая проблему "исчезающего градиента" (vanishing gradient problem), характерную для простых RNN.

**Зачем нужен LSTM?**

В задачах обработки текста, таких как анализ тональности, машинный перевод или генерация текста, важен контекст и порядок слов. LSTM способен улавливать эти зависимости, даже если слова, влияющие на смысл, находятся далеко друг от друга в предложении. Это делает LSTM мощным инструментом для понимания и обработки естественного языка.

**Как задается в TensorFlow (Keras)?**

LSTM слой в TensorFlow Keras задается с помощью класса `tf.keras.layers.LSTM`. Основные параметры:

*   `units`: Количество ячеек (нейронов) в слое LSTM. Это определяет размерность выходного вектора слоя.
*   `activation`: Функция активации для выходных данных (по умолчанию 'tanh').
*   `recurrent_activation`: Функция активации для рекуррентных шагов (по умолчанию 'sigmoid').
*   `use_bias`: Использовать ли смещение (bias).
*   `kernel_initializer`, `recurrent_initializer`, `bias_initializer`: Методы инициализации весов.
*   `dropout`: Вероятность отбрасывания нейронов на входе слоя (для регуляризации).
*   `recurrent_dropout`: Вероятность отбрасывания рекуррентных соединений (для регуляризации).



### SpatialDropout1D vs Dropout

Оба `Dropout` и `SpatialDropout1D` являются техниками регуляризации, используемыми для предотвращения переобучения нейронных сетей. Они делают это путем случайного "выключения" (обнуления) некоторых нейронов или элементов входных данных во время обучения. Однако, они применяют эту операцию по-разному, что делает `SpatialDropout1D` особенно полезным для последовательных данных, таких как текст.

**Dropout:**

*   **Как работает:** Случайным образом устанавливает в ноль отдельные элементы входного тензора с заданной вероятностью `rate`.
*   **Применение:** Обычно используется в полносвязных слоях (`Dense`).
*   **Проблема с последовательными данными:** При применении `Dropout` напрямую к выходу слоя Embedding или другим слоям, работающим с последовательностями (например, LSTM), он может случайно отбрасывать отдельные слова или их представления. Это может нарушить пространственные (временные) зависимости между соседними словами, которые важны для понимания контекста в тексте.

**SpatialDropout1D:**

*   **Как работает:** Случайным образом устанавливает в ноль целые каналы (например, целые признаки для всех слов в последовательности) входного тензора с заданной вероятностью `rate`. Для 1D данных (как в нашем случае с текстом), это означает, что если один признак (например, определенная размерность вектора эмбеддинга) отбрасывается для одного слова в последовательности, то он отбрасывается для *всех* слов в этой последовательности.
*   **Применение:** Особенно полезен после слоев Embedding или других слоев, работающих с последовательностями.
*   **Преимущество для последовательных данных:** `SpatialDropout1D` сохраняет пространственные (временные) корреляции между признаками. Вместо того чтобы отбрасывать отдельные слова, он отбрасывает целые признаки для всех слов. Это помогает предотвратить переобучение, сохраняя при этом целостность последовательных данных и их контекстуальные зависимости.

**В чем разница?**

Основное отличие заключается в том, что `Dropout` отбрасывает отдельные элементы, тогда как `SpatialDropout1D` отбрасывает целые *каналы* признаков по всем временным шагам (для 1D данных). Для текстов, где порядок и взаимосвязь слов важны, `SpatialDropout1D` часто дает лучшие результаты, поскольку он не разрушает локальные структуры в данных так сильно, как обычный `Dropout`.

# Задание 5

Задайте архитектуру LSTM модели в Tensorflow Keras, состяющую из следующих псоледовательно расположенных слоёв:


*   Embedding - output_dim = 16
*   Bidirectional LSTM - 32 нейрона
*   Полносвязный Dense - 10 нейронов и функция активации ReLU
*   Полносвязный Dense - 1 нейрон и функция активации sigmoid

У нас задача бинарной классификации, оптимизатор Adam, метрика - точность.

Обучите полученную модель.

Дополнительно:* При желании попробуйте поэкспериментировать с архитектурой и гиперпараметрами.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, SpatialDropout1D, Embedding, BatchNormalization, Bidirectional


#### ВСТАВЬТЕ КОД СЮДА

embedding_vector_length = 16

model_tf = ...

####

In [ ]:
# Обучение модели
history = model_tf.fit(X_train, y_train, validation_split=0.2, epochs=7)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.title('Точность модели')
plt.xlabel('Эпоха')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

# Анализ тональности с моделью BERT

Использование трансформеров в задаче анализа тональности считается лучшим подходом к решению на данный момент.

Вы можете попробовать использовать одну из предобученных BERT моделей для разметки своего датасета для курсовой в задаче анализа тональности.

In [ ]:
from transformers import pipeline
# Более лёгкая модель
model = pipeline(model="seara/rubert-tiny2-russian-sentiment")
# seara/rubert-base-cased-russian-sentiment - тяжёлая модель с большим количеством параетров и ожидаемыми хорошими результатами
model("Привет, ты мне нравишься!")

In [ ]:
# Применение модели тональности к 5 примерам из столбца 'text'
# import pandas as pd

if 'df' in locals() and 'text' in df.columns and not df.empty:
      # Примеры предсказаний
    print("\nПримеры предсказаний:")
    # Выбираем примеры случайным образом из тестовой выборки для демонстрации
    example_indices = df['processed_text'].sample(5, random_state=42).index
    examples = df.loc[example_indices]
    examples['predicted_sentiment'] = model(examples['text'].tolist())
    examples['predicted_sentiment'] = examples['predicted_sentiment'].apply(lambda x: x['label'])

    for index, row in examples.iterrows():
        print(f"\nОригинальный отзыв: {row['text']}")
        # print(f"Предобработанный отзыв: {row['processed_text']}")
        print(f"Истинная тональность: {row['sentiment']}")
        print(f"Предсказанная тональность (BERT): {row['predicted_sentiment']}")


In [ ]:
examples

In [ ]:
def bert_model(text):
    if isinstance(text, str):
        return model(text)[0]['label']
    return "" # Возвращаем пустую строку для нестроковых значений

print("Начинаем предсказывать")
tqdm.pandas()
df['predicted_sentiment'] = df['processed_text'].progress_apply(bert_model)
print("Закончили предсказывать.")
display(df.head())

In [ ]:
df

In [ ]:
precision, recall, f1, _ = precision_recall_fscore_support(df['sentiment'], df['predicted_sentiment'], average=None, labels=all_labels, zero_division=0) # Расчет по классам
accuracy = accuracy_score(df['sentiment'], df['predicted_sentiment'])

In [ ]:
print("\nРезультаты оценки модели:")
print(f"Общая точность (Accuracy): {accuracy:.4f}")

# Вывод метрик по классам
metrics_df = pd.DataFrame({
    'Sentiment': all_labels,
    'Precision': precision,
    'Recall': recall,
    'F1-score': f1
})
print("\nМетрики по классам:")
display(metrics_df)

In [ ]:
df.iloc[2]['text']

In [ ]:
df.iloc[2]

Какие выводы можно сделать на основании полученных метрик?

Подумайте о возможных подходах к улучшению точности на этапе предварительной обработки, токенизации и векторизации текста (использование биграмм, корректирование списка стоп-слов, ...).